# Local and open models
**Ollama in depth · where open weights come from · quantization · tool calling on small models · local vs hosted speed**

Before starting: `git pull`, `uv sync` (no new packages). You'll need Ollama running with at least two local models pulled — see the pre-read.

## 1. Setup

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from local_models_utils import model_size_gb, fits_on, laptop_options, load_local_catalog, probe_tool_calling, speed_table
from providers_utils import groq_client, ollama_client, ask
from landscape_utils import time_stream, PROMPT

groq = groq_client()
GROQ_MODEL = os.getenv("MODEL")
try:
    ollama = ollama_client()
    local_models = [m.id for m in ollama.models.list().data]
    print("Ollama models available:", local_models)
except Exception as e:
    ollama, local_models = None, []
    print("Ollama not reachable:", e)

## 2. Where open models come from: Hugging Face
Ollama does not train models — it packages and runs models published on hubs like Hugging Face, the same place Groq's hosted open-weight models originate from. The names differ slightly: a model on Hugging Face as `meta-llama/Llama-3.2-3B-Instruct` becomes `llama3.2:3b` as an Ollama tag, and `openai/gpt-oss-20b` on Groq is the same underlying weights as `gpt-oss:20b` if you pulled it locally.

*Question 1 — If a Hugging Face model page says "gated: request access", what does that tell you about its licence?*

In [ ]:
print(load_local_catalog())

## 3. Quantization: what it actually costs in memory
Every row below is the same family of maths you saw in Session 10: parameters x bytes-per-parameter. The only new idea is precision — how many bytes each parameter uses.

In [ ]:
print(f"{'precision':10} {'bytes/param':>12} {'a 20B model':>14}")
for prec in ("fp32", "fp16", "int8", "int4"):
    print(f"{prec:10} {'':>12} {model_size_gb(20, prec):>12.1f} GB")

In [ ]:
# Which catalog models fit YOUR laptop? Edit this to your real RAM.
MY_RAM_GB = 16
fits = laptop_options(MY_RAM_GB)
print(f"On a {MY_RAM_GB} GB laptop, these fit (with 2 GB headroom for everything else):")
for m in fits:
    print(f"  {m['name']:22} {model_size_gb(m['params_billion'], m['precision']):5.1f} GB   tool calling: {m['tool_calling']}")

*Question 2 — Why does the table use int4 sizes for the catalog instead of fp16, when fp16 is what the model was trained at?*

## 4. Does a small local model support tool calling?
Not every small model handles function calling reliably — some will just talk about calling the tool instead of actually emitting a structured call. This matters directly for Module 6: an agent that can't reliably call tools can't be an agent.

In [ ]:
if ollama and local_models:
    for name in local_models[:3]:
        r = probe_tool_calling(ollama, name)
        print(f"{name:20} supports_tool_calling={r['supports_tool_calling']}")
        if not r["supports_tool_calling"]:
            print("   said instead:", (r.get('raw_text') or '')[:150])
else:
    print("skipped (Ollama not running)")

*Question 3 — If your local model fails the tool-calling probe, what are your two options for Module 6's agent labs?*

## 5. Local vs hosted: measure, don't guess
Same prompt, same checkable-answer question from Session 10, sent to Groq and to your local model. Read your own numbers — they depend on your laptop, not on a published benchmark.

In [ ]:
results = {}
try:
    results["groq"] = time_stream(groq, GROQ_MODEL)
except Exception as e:
    print("groq failed:", e)
if ollama and local_models:
    try:
        results["ollama"] = time_stream(ollama, local_models[0])
    except Exception as e:
        print("ollama failed:", e)
else:
    print("ollama skipped (not running)")

if results:
    print(speed_table(results))

*Question 4 — Your laptop's tokens/second is probably 5-15x slower than Groq's. For which of the three jobs from Session 10 (FAQ bot, document reader, agent-assist) does that gap matter least, and why?*

## 6. Reflection
Answer questions 1-4 above, one or two lines each.

1.
2.
3.
4.

Then run the checkpoint: `uv run pytest tests/test_s12.py`